<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Responsible AI & Deployment (Capstone)** · Week 3 · Fairness, Explainability, Privacy and Security · Lesson 13 of 24

# Measuring Fairness

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/responsible-ai-deployment).

**Learning objective:** Measure how a model treats different groups with approval rates, true and false positive rates, disparate impact and equal opportunity, and explain why different fairness definitions can disagree.

## Why measure fairness?

A model can be accurate **on average** and still treat a group of people worse — rural applicants, women, older people, speakers of one language. In lending, that means some people are refused credit they would have repaid. You cannot fix what you do not measure.

Our model does **not** use gender or area as inputs. Does that make it fair? Let us measure.

In [ ]:
import numpy as np
import pandas as pd

def make_applicants(n=3000, seed=7, drift=0.0, concept=0.0):
    # Made-up SACCO loan applications. drift > 0: incomes, loans and late payments change.
    # concept > 0: the link to repayment changes (salaried jobs protect less, late payments hurt more).
    rng = np.random.default_rng(seed)
    gender = rng.choice(["female", "male"], n)
    area = rng.choice(["urban", "rural"], n, p=[0.6, 0.4])
    rural = area == "rural"
    income = (rng.lognormal(12.4, 0.5, n) * np.where(rural, 0.7, 1.0) * (1 - drift * 0.25)).round(-3)
    loan = (rng.uniform(50_000, 600_000, n) * (1 + drift)).round(-3)
    years = rng.integers(0, 13, n)
    late = rng.poisson(0.7 + drift, n)
    salaried = (rng.random(n) < np.where(rural, 0.3, 0.6)).astype(int)
    momo = np.clip(rng.normal(np.where(rural, 18, 45), 10), 0, None).round().astype(int)
    score = 3.0 - 1.6 * (loan / income) + 0.12 * years - 0.7 * late + 0.6 * salaried + 0.01 * momo
    score = score - concept * (1.5 * salaried + 0.6 * late - 0.04 * (momo - 30))
    repaid = (rng.random(n) < 1 / (1 + np.exp(-score))).astype(int)
    officer = score - 0.9 * rural - 0.6 * (gender == "female") + rng.normal(0, 0.6, n)
    return pd.DataFrame({
        "applicant_id": np.arange(1, n + 1), "age": rng.integers(20, 63, n), "gender": gender, "area": area,
        "income_rwf": income, "loan_rwf": loan, "years_member": years, "late_payments": late,
        "salaried": salaried, "momo_tx_month": momo,
        "approved_before": (officer > 1.2).astype(int), "repaid": repaid})

applicants = make_applicants()
print(applicants.shape, "| repaid:", round(applicants["repaid"].mean(), 3))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ["income_rwf", "loan_rwf", "years_member", "late_payments", "salaried", "momo_tx_month"]
TARGET = "repaid"
train, test = train_test_split(applicants, test_size=0.25, random_state=0, stratify=applicants[TARGET])
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(train[FEATURES], train[TARGET])
proba = model.predict_proba(test[FEATURES])[:, 1]
print(f"accuracy {accuracy_score(test[TARGET], proba >= 0.5):.3f} | AUC {roc_auc_score(test[TARGET], proba):.3f}")

## Group metrics

In [ ]:
def group_report(data, group, decision, outcome="repaid"):
    rows = []
    for name, g in data.groupby(group):
        rows.append({
            group: name, "people": len(g),
            "approval_rate": g[decision].mean(),                              # selection rate
            "approved_if_would_repay": g.loc[g[outcome] == 1, decision].mean(), # true positive rate
            "approved_if_would_default": g.loc[g[outcome] == 0, decision].mean(), # false positive rate
            "repay_rate": g[outcome].mean(),
        })
    return pd.DataFrame(rows).set_index(group).round(3)

def fairness_summary(report):
    a, t = report["approval_rate"], report["approved_if_would_repay"]
    return {"approval_gap": round(float(a.max() - a.min()), 3),
            "disparate_impact": round(float(a.min() / a.max()), 3),
            "equal_opportunity_gap": round(float(t.max() - t.min()), 3)}

In [ ]:
audit = test.copy()
audit["approved"] = (model.predict_proba(audit[FEATURES])[:, 1] >= 0.5).astype(int)

for group in ["area", "gender"]:
    report = group_report(audit, group, "approved")
    print(report.to_string()); print(fairness_summary(report), "\n")

What each column means:

- **approval_rate** — share of the group approved (also called selection rate).
- **approved_if_would_repay** — among people who really would repay, the share approved (the **true positive rate**). Low values mean good borrowers are being refused.
- **approved_if_would_default** — among people who would default, the share approved (the **false positive rate**).
- **repay_rate** — how often the group really repays.

## Three fairness measures

| Measure | Question | Our model, by area |
|---|---|---|
| **Approval gap** (demographic parity difference) | Are groups approved at similar rates? | Large |
| **Disparate impact ratio** | Lowest approval rate ÷ highest | Below 0.8 |
| **Equal opportunity gap** | Are people who would repay approved at similar rates? | Several points |

The "**four-fifths rule**" from US employment guidelines treats a disparate-impact ratio below **0.8** as a warning sign. It is a rule of thumb, not a law of nature — but a useful alarm.

For **gender**, all measures are small: women and men repay at the same rate and the model treats them alike. For **area**, rural applicants are approved much less often.

## Is that unfair? It depends on the definition

Rural applicants in our data **do** repay less often (lower incomes, fewer salaried jobs). So:

- By **demographic parity**, the model is unfair: approval rates differ a lot.
- By **equal opportunity**, it is partly unfair: rural people who **would** repay are still refused more often than urban people who would repay.
- By **calibration** (a 70% score means 70% repay, in every group), it may be fair.

Researchers have proved that when groups have different repay rates, these definitions **cannot all be satisfied at once** (except with a perfect model). Choosing the definition is not a technical choice: it is a decision about values, made with the SACCO's leaders, the people affected, and the law — and written down (Week 4).

In [ ]:
import matplotlib.pyplot as plt
report = group_report(audit, "area", "approved")
report[["approval_rate", "approved_if_would_repay", "repay_rate"]].plot.bar(figsize=(7, 3.5), rot=0)
plt.title("Loan model by area"); plt.ylim(0, 1); plt.show()

## Good habits

- Measure fairness for **every group** that matters — even when the group is not a model input.
- Keep group labels (gender, area) in a **separate, protected** table used only for auditing.
- Look at **several** measures, and at small groups' sample sizes (a rate from 20 people is noisy).
- Report results honestly, including where the model is unfair.

## Key terms

- **Selection rate** — share of a group that gets the positive decision.
- **True positive rate** — share of truly positive cases that are approved.
- **Demographic parity** — equal selection rates across groups.
- **Disparate impact ratio** — lowest selection rate ÷ highest; below 0.8 is a warning.
- **Equal opportunity** — equal true positive rates across groups.
- **Calibration** — scores mean the same probability in every group.

**Takeaway:** Leaving gender and area out of the model does not guarantee fairness; measure group metrics, know that fairness definitions conflict, and decide the definition openly with the people affected.

## Try it yourself

1. Add an age group column (under 30, 30–45, over 45) and run `group_report` for it.
2. Change the approval threshold to 0.6. What happens to the area gaps?
3. Check calibration: for applicants with scores between 0.6 and 0.8, what share repaid in each area?
4. Write two sentences for the SACCO board explaining the difference between demographic parity and equal opportunity.

---
Finished? Go back to [Responsible AI & Deployment (Capstone) on WAMUGII](https://wamugii.kento.rw/courses/responsible-ai-deployment), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*